# Procesamiento de citas SDR/LQT — Bold

Lee un CSV grande por bloques, limpia fechas y TPV, calcula KPIs y exporta tablas compactas para Sites, n8n, Slack y Gemini.


In [ ]:
SOURCE_PATH = '/Users/juliandavidoviedopasuy/Documents/Bold/Copia de Citas SDR _ LQT - Respuestas del formulario.csv'
SOURCE_URL = ''  # Pega aquí la URL CSV publicada para usarla como fuente
OUTPUT_DIR = 'sdr_outputs'
START_DATE, END_DATE = '2025-08-01', '2025-09-30'
PREVENTIVE_DAYS, CRITICAL_DAYS = 3, 5

import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
Path(OUTPUT_DIR).mkdir(exist_ok=True)


## Carga y normalización

Se procesa por bloques para evitar cargar innecesariamente el archivo completo en memoria.


In [ ]:
source = SOURCE_URL.strip() or SOURCE_PATH
chunks = []
for chunk in pd.read_csv(source, dtype=str, chunksize=5000, encoding='utf-8-sig', on_bad_lines='warn'):
    chunks.append(chunk)
raw = pd.concat(chunks, ignore_index=True)
print(f'Filas: {len(raw):,} | Columnas: {len(raw.columns):,} | Fuente: {source}')

def parse_date(series):
    s = series.astype(str).str.strip()
    return pd.to_datetime(s, dayfirst=True, errors='coerce')

def money(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[^0-9-]', '', regex=True), errors='coerce').fillna(0)

df = raw.copy()
df['fecha_cita'] = parse_date(df['Fecha de la cita'])
df['fecha_agendo'] = parse_date(df['Fecha agendo'])
df['fecha_cierre'] = parse_date(df['Fecha de cierre venta'])
df['fecha_ultima_act_op'] = parse_date(df['Dia Ultima Act OP'])
for src, dst in [('TPV M0','tpv_m0'),('TPV M1','tpv_m1'),('TPV M2','tpv_m2')]: df[dst] = money(df[src])
df['es_futuro'] = df.fecha_cita > pd.Timestamp.today().normalize()


## Filtro, llaves y KPIs


In [ ]:
period = df[(df.fecha_cita >= pd.Timestamp(START_DATE)) & (df.fecha_cita <= pd.Timestamp(END_DATE)) & (~df.es_futuro)].copy()
period['id_cita'] = period['ID del evento'].replace({'':np.nan, 'nan':np.nan, '#N/A':np.nan})
period['id_cita'] = period['id_cita'].fillna(period['Documento NIT / CC'].astype(str)+'|'+period.fecha_cita.dt.strftime('%Y-%m-%d')+'|'+period['Ejecutivo asignado'].astype(str))
period['id_comercio'] = period['Merchant'].where(~period['Merchant'].isin(['','-','#N/A']), period['Documento NIT / CC'])
period['oportunidad'] = pd.to_numeric(period['Conteo OP'], errors='coerce').fillna(0).eq(1)
period['cierre'] = pd.to_numeric(period['Conteo Cierre'], errors='coerce').fillna(0).eq(1)
period['pendiente'] = (~period['cierre']) & (~period['Ultimo estado OP'].isin(['Perdida']))
period['semana'] = period.fecha_cita.dt.to_period('W').astype(str)
weekly = period.groupby('semana').agg(citas=('id_cita','nunique'), oportunidades=('oportunidad','sum'), cierres=('cierre','sum'), pendientes=('pendiente','sum'), comercios=('id_comercio','nunique')).reset_index()
weekly['conversion_op_sobre_citas'] = np.where(weekly.citas>0, weekly.oportunidades/weekly.citas, np.nan)
weekly['conversion_cierre_sobre_op'] = np.where(weekly.oportunidades>0, weekly.cierres/weekly.oportunidades, np.nan)
display(weekly)


## Gestión, alertas y exportación

La alerta real requiere una fecha diaria de transacción. `Dia Ultima Act OP` se usa sólo como aproximación y debe validarse.


In [ ]:
exec_summary = period.groupby(['Ejecutivo asignado','Team Lead','Canal actual de la venta'], dropna=False).agg(citas=('id_cita','nunique'), oportunidades=('oportunidad','sum'), cierres=('cierre','sum'), pendientes=('pendiente','sum'), tpv_m0=('tpv_m0','sum'), tpv_m1=('tpv_m1','sum')).reset_index()
exec_summary['Team Lead'] = exec_summary['Team Lead'].replace({'#N/A':'Responsable pendiente','':'Responsable pendiente'})
cutoff = pd.Timestamp(END_DATE)
alerts = period[period.cierre].copy()
alerts['dias_sin_transar'] = (cutoff - alerts.fecha_ultima_act_op).dt.days
alerts['nivel_alerta'] = np.select([alerts.dias_sin_transar>=CRITICAL_DAYS, alerts.dias_sin_transar>=PREVENTIVE_DAYS], ['Crítica','Preventiva'], default='Normal')
alerts = alerts[alerts.nivel_alerta != 'Normal']
alert_cols = ['id_comercio','Empresa','fecha_cierre','fecha_ultima_act_op','dias_sin_transar','nivel_alerta','Ejecutivo asignado','Team Lead','Canal actual de la venta','tpv_m0','tpv_m1']
alerts_out = alerts[alert_cols].sort_values(['nivel_alerta','dias_sin_transar','tpv_m1'], ascending=[True,False,False]).drop_duplicates(['id_comercio','nivel_alerta'])
quality = pd.DataFrame({'campo':raw.columns, 'nulos_pct':[round(raw[c].isna().mean()*100,2) for c in raw.columns], 'errores_NA':[(raw[c].astype(str).str.strip()=='#N/A').sum() for c in raw.columns]})
weekly.to_csv(f'{OUTPUT_DIR}/kpis_semanales.csv', index=False)
exec_summary.to_csv(f'{OUTPUT_DIR}/gestion_ejecutivos.csv', index=False)
alerts_out.to_csv(f'{OUTPUT_DIR}/alertas_comercios.csv', index=False)
quality.to_csv(f'{OUTPUT_DIR}/calidad_datos.csv', index=False)
print(sorted(os.listdir(OUTPUT_DIR)))
